# 09 · Store points: ids, vectors and payloads
Level L2 · Part 2 · builds on lesson 08 (and lesson 07's first points)

## Goal
By the end of this lesson you can load all 50 recipes into the `recipes` collection as points,
each with a stable id, its vector and a payload of typed metadata, and read them back by id,
choosing which parts come back.

## The idea
A **point** is one stored item: an **id**, a **vector** and a **payload**. The id is the
recipe's permanent name in the database: a whole number (0 or more) or a UUID. The vector is
what search compares. The payload is a small JSON object of facts you want back with each hit
and, from lesson 11, to filter on. You write points with **upsert**: insert the point, or
replace it completely if that id already exists. Think of a library: the id is the shelf mark,
the vector is where the book sits by topic, the payload is the index card.

## Picture

```mermaid
flowchart LR
    R["CSV row<br/>id 1, Tomato soup,<br/>Italian, true, 35"] --> P
    E["doc_emb[0]<br/>384 numbers"] --> P
    P["PointStruct<br/>id = 1<br/>vector = 384 numbers<br/>payload = {title, cuisine,<br/>vegetarian, minutes, embed_model}"]
    P -- "upsert" --> C[("collection 'recipes'<br/>size 384, Cosine")]
    C -- "retrieve(ids=[1])<br/>with_payload / with_vectors" --> B["Record(id=1, payload=..., vector=...)"]
```

A row of the CSV and a row of the embeddings become one point; `retrieve` gives it back by id.

## Step by step
### Step 1 · Data and an empty `recipes` collection
Same data as lesson 08: 50 recipes and their MiniLM embeddings (row `i` of `doc_emb` belongs to
row `i` of the CSV). We open this lesson's own local folder and create `recipes` with the
settings lesson 08 chose, reading the size from the data.

In [1]:
import shutil, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message="IProgress")     # quiet a progress-bar notice on import
from qdrant_client import QdrantClient, models

recipes = pd.read_csv("../data/recipes_50.csv")
saved = np.load("../data/recipe_embeddings_minilm.npz")
doc_emb, MODEL = saved["doc_emb"], str(saved["model"])
DB_PATH = "../data/cache/qdrant_09"
shutil.rmtree(DB_PATH, ignore_errors=True)           # start from an empty folder every run
client = QdrantClient(path=DB_PATH)
client.create_collection("recipes", vectors_config=models.VectorParams(
    size=doc_emb.shape[1], distance=models.Distance.COSINE))
print("model:", MODEL, "| doc_emb:", doc_emb.shape, "| points:", client.count("recipes").count)

model: sentence-transformers/all-MiniLM-L6-v2 | doc_emb: (50, 384) | points: 0


An empty collection of size 384, Cosine, exactly as lesson 08 left it.

### Step 2 · One point by hand
Start with the smallest thing: Tomato soup alone. A `PointStruct` holds the three parts. The
vector must be a plain list of 384 numbers; the payload a dict.

In [2]:
first = recipes.iloc[0]
soup = models.PointStruct(
    id=1,
    vector=doc_emb[0].tolist(),
    payload={"title": "Tomato soup", "cuisine": "Italian", "vegetarian": True, "minutes": 35},
)
print("id:", soup.id, "| vector:", len(soup.vector), "numbers, first 3", np.round(soup.vector[:3], 3))
print("payload:", soup.payload)
print("CSV row 0 says:", first["id"], first["title"], first["cuisine"], first["vegetarian"], first["minutes"])

id: 1 | vector: 384 numbers, first 3 [-0.144  0.015  0.034]
payload: {'title': 'Tomato soup', 'cuisine': 'Italian', 'vegetarian': True, 'minutes': 35}
CSV row 0 says: 1 Tomato soup Italian True 35


The point matches the CSV row. Nothing is stored yet; `upsert` sends it.

In [3]:
result = client.upsert(collection_name="recipes", points=[soup])
print(result)
print("points in collection:", client.count("recipes").count)

operation_id=0 status=<UpdateStatus.COMPLETED: 'completed'>
points in collection: 1


`status=completed` means the point is written before `upsert` returns (the default `wait=True`;
lesson 14 discusses not waiting). `operation_id` is a sequence number for the write; local mode
always prints 0. One point stored.

### Step 3 · Read it back by id
`retrieve` fetches points by id. By default it returns the payload but **not** the vector,
because vectors are large and usually not needed for display.

In [4]:
records = client.retrieve(collection_name="recipes", ids=[1])
rec = records[0]
print(type(rec).__name__, "| id:", rec.id)
print("payload:", rec.payload)
print("vector:", rec.vector)

Record | id: 1
payload: {'title': 'Tomato soup', 'cuisine': 'Italian', 'vegetarian': True, 'minutes': 35}
vector: None


A `Record` with `vector: None`. Ask for the vector explicitly with `with_vectors=True` and
compare it to what we sent:

In [5]:
rec = client.retrieve(collection_name="recipes", ids=[1], with_vectors=True)[0]
back = np.array(rec.vector, dtype=np.float32)
print("vector length:", len(back))
print("same as doc_emb[0]?", np.allclose(back, doc_emb[0], atol=1e-6))
print("largest difference:", float(np.abs(back - doc_emb[0]).max()))

vector length: 384
same as doc_emb[0]? True
largest difference: 0.0


The same 384 numbers come back (MiniLM vectors are already unit length, so Cosine's
normalising in lesson 08 changed nothing measurable).

### Step 4 · Ids: whole numbers or UUIDs
Qdrant's docs allow two id types: **64-bit unsigned integers** (0, 1, 2, …) and **UUIDs**
(a 128-bit identifier written like `550e8400-e29b-41d4-a716-446655440000`). Our CSV already
has a stable integer `id`, so we use it. A UUID suits data whose natural key is text, such as a
URL. Try one in a throwaway in-memory collection:

In [6]:
import uuid

lab = QdrantClient(":memory:")
lab.create_collection("scratch", vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE))
web_id = str(uuid.uuid5(uuid.NAMESPACE_URL, "https://example.com/recipes/tomato-soup"))
lab.upsert("scratch", points=[models.PointStruct(id=web_id, vector=doc_emb[0].tolist(), payload={"title": "Tomato soup"})])
print("UUID id:", web_id)
print(lab.retrieve("scratch", ids=[web_id]))

UUID id: a0f01ca9-261c-5afc-95a7-67f2092b3341
[Record(id='a0f01ca9-261c-5afc-95a7-67f2092b3341', payload={'title': 'Tomato soup'}, vector=None, shard_key=None, order_value=None)]


`uuid5` turns the same URL into the same UUID every time, so reloading the page's recipe hits
the same point (lesson 14 builds on this). A text id like `"recipe-1"` is *not* allowed: see
Common mistakes.

### Step 5 · Design the payload
The payload holds what a search result needs to show and what you will filter on. For a recipe:
`title` (display), `cuisine`, `vegetarian`, `minutes` (filters in lesson 11), plus the
`embed_model`, so a program can check which model made the vector (lesson 08 found the size
check cannot). Types matter: a filter on `minutes <= 30` needs a number, not the text `"35"`.

In [7]:
def to_payload(row):
    """One CSV row -> a payload of plain JSON types."""
    return {
        "title": str(row.title),
        "cuisine": str(row.cuisine),
        "vegetarian": bool(row.vegetarian),
        "minutes": int(row.minutes),
        "embed_model": MODEL,
    }

row0 = next(recipes.itertuples())
print({k: (v, type(v).__name__) for k, v in to_payload(row0).items()})

{'title': ('Tomato soup', 'str'), 'cuisine': ('Italian', 'str'), 'vegetarian': (True, 'bool'), 'minutes': (35, 'int'), 'embed_model': ('sentence-transformers/all-MiniLM-L6-v2', 'str')}


Each field has a JSON type: text (Qdrant calls it *keyword*), bool and integer. The
`description` stays in the CSV for now; long text can live in the payload too, but keep payloads
to what you show or filter.

### Step 6 · Upsert all 50 recipes
Build one `PointStruct` per row, pairing CSV row `i` with `doc_emb[i]`, and send them in one call.
Tomato soup (id 1) is in the list again.

In [8]:
points = [
    models.PointStruct(id=int(row.id), vector=doc_emb[i].tolist(), payload=to_payload(row))
    for i, row in enumerate(recipes.itertuples())
]
print("built", len(points), "points; ids", points[0].id, "...", points[-1].id)
print(client.upsert(collection_name="recipes", points=points))
print("points in collection:", client.count("recipes").count)

built 50 points; ids 1 ... 50


operation_id=0 status=<UpdateStatus.COMPLETED: 'completed'>
points in collection: 50


50 points, not 51. Tomato soup's id 1 already existed, so upsert **replaced** it rather than
adding a copy; its payload now also has `embed_model`. That makes loading **idempotent**:
running the same load twice leaves the same 50 points.

In [9]:
client.upsert(collection_name="recipes", points=points)
print("after loading a second time:", client.count("recipes").count, "points")
print(client.retrieve(collection_name="recipes", ids=[1])[0].payload)

after loading a second time: 50 points
{'title': 'Tomato soup', 'cuisine': 'Italian', 'vegetarian': True, 'minutes': 35, 'embed_model': 'sentence-transformers/all-MiniLM-L6-v2'}


Still 50, and id 1 carries the full five-field payload.

### Step 7 · Choose what comes back
`with_payload` accepts `True`, `False`, or a list of field names; `with_vectors` accepts `True`
or `False`. Fetch three recipes with only the two fields a results list needs:

In [10]:
light = client.retrieve(collection_name="recipes", ids=[2, 5, 9], with_payload=["title", "minutes"])
for r in light:
    print(r.id, r.payload, "| vector:", r.vector)
bare = client.retrieve(collection_name="recipes", ids=[2], with_payload=False)
print("with_payload=False:", bare[0])

2 {'title': 'Mango salsa', 'minutes': 10} | vector: None
5 {'title': 'Lemon sorbet', 'minutes': 20} | vector: None
9 {'title': 'French onion soup', 'minutes': 75} | vector: None
with_payload=False: id=2 payload=None vector=None shard_key=None order_value=None


Only `title` and `minutes` came back; with `with_payload=False` the payload is `None` and you get
the id alone. Sending less is how real apps keep responses small.

### Step 8 · Ids that do not exist
What if you ask for an id that is not there, say 999?

In [11]:
found = client.retrieve(collection_name="recipes", ids=[3, 999, 4])
print("asked for 3 ids, got", len(found), "records:", [(r.id, r.payload["title"]) for r in found])
missing = {3, 999, 4} - {r.id for r in found}
print("missing ids:", missing)

asked for 3 ids, got 2 records: [(3, 'Chocolate brownies'), (4, 'Caesar salad')]
missing ids: {999}


No error: missing ids are simply left out. Compare what you asked for with what came back
whenever a missing point matters (for example, a saved favourite that was deleted).

### Step 9 · Upsert replaces the whole point
Upsert is not "update some fields". If you upsert id 1 with a payload that only has a title,
the other fields are gone, because the old point is deleted and the new one inserted:

In [12]:
client.upsert("recipes", points=[models.PointStruct(id=1, vector=doc_emb[0].tolist(), payload={"title": "Tomato soup"})])
print("after a title-only upsert:", client.retrieve("recipes", ids=[1])[0].payload)
client.upsert("recipes", points=[points[0]])                  # put the full point back
print("restored:", client.retrieve("recipes", ids=[1])[0].payload)

after a title-only upsert: {'title': 'Tomato soup'}
restored: {'title': 'Tomato soup', 'cuisine': 'Italian', 'vegetarian': True, 'minutes': 35, 'embed_model': 'sentence-transformers/all-MiniLM-L6-v2'}


To change one field without resending everything, lesson 12 uses `set_payload`. For a full load
like Step 6, replacing is exactly what you want.

## What just happened
- Step 2 built one `PointStruct` (id 1, 384 numbers, 4-field payload); upsert returned `status=completed` and the count went to 1.
- Step 3: `retrieve` returned the payload with `vector: None`; `with_vectors=True` gave back the exact 384 numbers.
- Step 4: a UUID made by `uuid5` from a URL was accepted as an id.
- Steps 5–6: `to_payload` gave plain JSON types; upserting 50 points (Tomato soup again) left 50, and a second load still 50.
- Steps 7–8: `with_payload=["title", "minutes"]` trimmed the payload; id 999 was silently missing.
- Step 9: a title-only upsert wiped the other fields until the full point was put back.

## Common mistakes

**1. Using a text label as the id.** `"recipe-1"` looks like a fine id, but it is neither an
unsigned integer nor a UUID:

In [13]:
try:
    client.upsert("recipes", points=[
        models.PointStruct(id="recipe-1", vector=doc_emb[0].tolist(), payload=to_payload(row0))])
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValueError: Point id recipe-1 is not a valid UUID


It prints `ValueError: Point id recipe-1 is not a valid UUID`. Fix: use the integer `id` you
already have, or turn the label into a UUID with `uuid5` (Step 4) and keep the label in the
payload.

In [14]:
label = "recipe-1"
fixed_id = str(uuid.uuid5(uuid.NAMESPACE_URL, label))
lab.upsert("scratch", points=[models.PointStruct(id=fixed_id, vector=doc_emb[0].tolist(), payload={"label": label})])
print(fixed_id, "->", lab.retrieve("scratch", ids=[fixed_id])[0].payload)

0015852d-4964-5dc5-a005-4be45e6b3bd0 -> {'label': 'recipe-1'}


**2. NumPy numbers in the payload.** `recipes.iloc[i]` returns NumPy values (`numpy.int64`,
`numpy.bool`), not Python ones. They look the same when printed, but JSON cannot hold them.
We try it in a throwaway collection: in local mode a failed write can leave a half-written point.

In [15]:
try:
    row = recipes.iloc[0]
    print(type(row["minutes"]), type(row["vegetarian"]))
    lab.create_collection("throwaway", vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE))
    lab.upsert("throwaway", points=[models.PointStruct(
        id=1, vector=doc_emb[0].tolist(), payload={"title": row["title"], "minutes": row["minutes"]})])
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

<class 'numpy.int64'> <class 'numpy.bool'>
PydanticSerializationError: Unable to serialize unknown type: <class 'numpy.int64'>


After printing the two NumPy types it prints
`PydanticSerializationError: Unable to serialize unknown type: <class 'numpy.int64'>`.
Fix: convert explicitly with `int()`, `bool()`, `str()` as `to_payload` does (or use
`itertuples()` / `to_dict("records")`, which give Python types).

In [16]:
safe = {"title": str(row["title"]), "minutes": int(row["minutes"]), "vegetarian": bool(row["vegetarian"])}
lab.upsert("scratch", points=[models.PointStruct(id=1, vector=doc_emb[0].tolist(), payload=safe)])
print({k: type(v).__name__ for k, v in lab.retrieve("scratch", ids=[1])[0].payload.items()})

{'title': 'str', 'minutes': 'int', 'vegetarian': 'bool'}


**3. Expecting the vector without asking for it.** Code that compares a stored vector to
something else fails, because `retrieve` leaves vectors out by default (Step 3):

In [17]:
try:
    stored = client.retrieve("recipes", ids=[2])[0]
    print("vector length:", len(stored.vector))
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

TypeError: object of type 'NoneType' has no len()


It prints `TypeError: object of type 'NoneType' has no len()`. Fix: pass `with_vectors=True`
when you need the numbers.

In [18]:
stored = client.retrieve("recipes", ids=[2], with_vectors=True)[0]
print("vector length:", len(stored.vector))
print(stored.payload["title"], "matches doc_emb[1]?", np.allclose(stored.vector, doc_emb[1], atol=1e-6))

vector length: 384
Mango salsa matches doc_emb[1]? True


## Try it
1. **Add a description.** Change `to_payload` to include the `description` column, reload all
   50 points, and confirm the count is still 50 and that recipe 7's payload now has 6 fields.
2. **Column-oriented upload.** `upsert` also accepts `models.Batch(ids=[...], vectors=[...],
   payloads=[...])`: three parallel lists instead of a list of points. Load recipes 1–10 that way
   into a new in-memory collection and retrieve ids 1 and 10 with only their titles.

In [19]:
# your code here

Close both clients so the local folder is released.

In [20]:
lab.close()
client.close()
print("closed")

closed


## Key terms
- **point id** — a point's permanent identifier: a 64-bit unsigned integer or a UUID; text labels are not accepted.
- **UUID** — a 128-bit identifier written as 32 hex digits (e.g. `550e8400-e29b-41d4-a716-446655440000`); `uuid5` makes the same UUID from the same text every time.
- **upsert** — write a point: insert it if its id is new, otherwise replace the whole point (vector and payload).
- **idempotent load** — a load that gives the same result however many times it runs, because points are keyed by stable ids.
- **retrieve** — read points by id; returns the payload by default, the vector only with `with_vectors=True`, and skips missing ids.
- **payload selector** — the `with_payload` argument (`True`, `False` or a list of field names) that chooses which payload fields come back.